In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [7]:
import pandas as pd
from src.config import (
    load_config,
    get_embedding_model,
    get_reranker_model,
    get_top_k
)
from src.knowledge_base import KnowledgeBaseService
from src.embeddings import EmbeddingService
from src.faiss_index import FaissIndexService
from src.retriever import RetrieverService
from src.reranker import RerankerService

In [3]:
knowledge_base_service = KnowledgeBaseService()
config = load_config()

embedding_service = EmbeddingService(
    get_embedding_model(config)
)

faiss_service = FaissIndexService()

retriever_service = RetrieverService(
    embedding_service,
    faiss_service,
)

reranker_service = RerankerService(
    get_reranker_model(config)
)

2026-07-12 12:48:26,846 | INFO | src.embeddings | Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-07-12 12:48:32,890 | INFO | src.embeddings | Embedding model loaded.
2026-07-12 12:48:32,894 | INFO | src.reranker | Loading Cross Encoder model: cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

2026-07-12 12:49:01,437 | INFO | src.reranker | Cross Encoder loaded successfully.


In [5]:
DATA_DIR = Path("../data/raw")
train = pd.read_csv(DATA_DIR / "train.csv")
kb = knowledge_base_service.build(train)

2026-07-12 12:50:13,842 | INFO | src.knowledge_base | Building Knowledge Base...
2026-07-12 12:50:13,859 | INFO | src.knowledge_base | Knowledge Base built successfully with 2000 documents.


In [6]:
faiss_service.load(
    PROJECT_ROOT
    / "outputs"
    / "faiss"
    / "knowledge_base.faiss"
)

2026-07-12 12:50:33,060 | INFO | src.faiss_index | FAISS index loaded successfully.


In [8]:
top_k = get_top_k(config)
query = "What is Machine Learning?"
distances, indices = retriever_service.retrieve(
    query,
    top_k,
)

2026-07-12 12:51:47,772 | INFO | src.retriever | Embedding query.
2026-07-12 12:51:47,774 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-12 12:51:48,868 | INFO | src.embeddings | Embedding generation completed.
2026-07-12 12:51:48,868 | INFO | src.retriever | Searching FAISS index.
2026-07-12 12:51:48,871 | INFO | src.retriever | Retrieved 10 documents.


In [9]:
results = reranker_service.rerank(
    query=query,
    retrieved_indices=indices[0].tolist(),
    knowledge_base=kb,
)

2026-07-12 12:52:06,086 | INFO | src.reranker | Reranking 10 retrieved documents.
2026-07-12 12:52:06,726 | INFO | src.reranker | Reranking completed.


In [10]:
for rank, result in enumerate(results, start=1):
    print(f"Rank {rank}")
    print(f"Index : {result['index']}")
    print(f"Score : {result['score']:.4f}")
    print(result["document"][:200])
    print("-" * 80)

Rank 1
Index : 225
Score : -6.0902
A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and inactivity-based forgetting.
--------------------------------------------------------------------------------
Rank 2
Index : 627
Score : -6.0902
A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and inactivity-based forgetting.
--------------------------------------------------------------------------------
Rank 3
Index : 725
Score : -6.0902
A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and inactivity-based forgetting.
--------------------------------------------------------------------------------
Rank 4
Index : 1096
Score : -6.0902
A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and inactivity-based forgetting.
--------------------------------------------------------------------------------

In [11]:
{
    "rank": 1,
    "index": 123,
    "document": "...",
    "score": 8.7421,
}

{'rank': 1, 'index': 123, 'document': '...', 'score': 8.7421}

In [12]:
for rank, result in enumerate(results, start=1):
    result["rank"] = rank